In [0]:
catalog = "logistics"
gold_schema = f"{catalog}.gold"

spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

orders_df = spark.table(
    "logistics.silver.shipment_orders"
)

tracking_df = spark.table(
    "logistics.silver.shipment_tracking_events"
)

current_df = spark.table(
    "logistics.silver.silver_shipment_current"
)

lifecycle_df = spark.table(
    "logistics.silver.silver_shipment_lifecycle"
)

summary_df = spark.table(
    "logistics.silver.silver_shipment_lifecycle_summary"
)

carrier_df = spark.table(
    "logistics.silver.carrier_master"
)

warehouse_df = spark.table(
    "logistics.silver.warehouse_master"
)

route_df = spark.table(
    "logistics.silver.route_master"
)

In [0]:
print("Orders:", orders_df.count())
print("Current status:", current_df.count())
print("Lifecycle:", lifecycle_df.count())
print("Lifecycle summary:", summary_df.count())
print("Carriers:", carrier_df.count())
print("Warehouses:", warehouse_df.count())
print("Routes:", route_df.count())

In [0]:
dim_carrier = carrier_df.select(
    "carrier_id",
    "carrier_name",
    "service_region",
    "carrier_tier",
    "contract_type",
    "service_level",
    "country",
    "active_flag"
).dropDuplicates(["carrier_id"])

display(dim_carrier)

In [0]:
dim_warehouse = warehouse_df.select(
    "warehouse_id",
    "warehouse_name",
    "city",
    "region",
    "country",
    "warehouse_type",
    "active_flag"
).dropDuplicates(["warehouse_id"])

display(dim_warehouse)

In [0]:
dim_route = route_df.select(
    "route_id",
    "origin_warehouse_id",
    "destination_city",
    "distance_km",
    "route_class",
    "planned_hours",
    "service_region",
    "vehicle_class"
).dropDuplicates(["route_id"])

display(dim_route)

In [0]:
display(
    orders_df.groupBy("shipment_id")
    .count()
    .filter(col("count") > 1)
)

In [0]:
fact_shipment_activity = (
    orders_df.alias("o")
    .join(
        summary_df.alias("s"),
        on="shipment_id",
        how="left"
    )
    .join(
        current_df.select(
            "shipment_id",
            col("event_type").alias("latest_tracking_status"),
            col("event_ts").alias("latest_event_ts")
        ).alias("c"),
        on="shipment_id",
        how="left"
    )
    .select(
        col("o.shipment_id"),
        col("o.order_id"),
        col("o.customer_id"),
        col("o.carrier_id"),
        col("o.origin_warehouse_id"),
        col("o.destination_city"),
        col("o.created_ts"),
        col("o.promised_delivery_ts"),
        col("o.package_weight_kg"),
        col("o.shipping_cost"),
        col("o.service_level"),
        col("o.shipment_status").alias("order_status"),
        col("s.pickup_time"),
        col("s.delivery_time"),
        col("s.delivery_duration_hours"),
        col("s.total_events"),
        col("s.delivery_observed"),
        col("c.latest_tracking_status"),
        col("c.latest_event_ts")
    )
)

display(fact_shipment_activity)

In [0]:
fact_shipment_activity = (
    fact_shipment_activity
    .join(
        dim_carrier.withColumnsRenamed({
            "service_region": "carrier_service_region",
            "service_level": "carrier_service_level",
            "country": "carrier_country",
            "active_flag": "carrier_active_flag"
        }),
        on="carrier_id",
        how="left"
    )
    .join(
        dim_warehouse.withColumnsRenamed({
            "warehouse_id": "origin_warehouse_id",
            "city": "warehouse_city",
            "region": "warehouse_region",
            "country": "warehouse_country",
            "active_flag": "warehouse_active_flag"
        }),
        on="origin_warehouse_id",
        how="left"
    )
    .join(
        dim_route.withColumnRenamed("service_region", "route_service_region"),
        on=["origin_warehouse_id", "destination_city"],
        how="left"
    )
)

In [0]:
display(fact_shipment_activity)

In [0]:
#Calculate on-time delivery
fact_shipment_activity = fact_shipment_activity.withColumn(
    "delivery_performance",
    when(
        col("delivery_time").isNull(),
        "NOT_DELIVERED"
    ).when(
        col("delivery_time") <= col("promised_delivery_ts"),
        "ON_TIME"
    ).otherwise("DELAYED")
)

In [0]:
display(
    fact_shipment_activity.groupBy(
        "delivery_performance"
    ).count()
)

In [0]:
on_time_kpi = fact_shipment_activity.agg(
    count(
        when(
            col("delivery_performance").isin(
                "ON_TIME", "DELAYED"
            ),
            1
        )
    ).alias("eligible_delivered_shipments"),

    count(
        when(
            col("delivery_performance") == "ON_TIME",
            1
        )
    ).alias("on_time_shipments")
)

on_time_kpi = on_time_kpi.withColumn(
    "on_time_delivery_rate",
    when(
        col("eligible_delivered_shipments") > 0,
        round(
            col("on_time_shipments") * 100 /
            col("eligible_delivered_shipments"),
            2
        )
    )
)

display(on_time_kpi)

In [0]:
#Calculate carrier lead time
carrier_lead_time = (
    fact_shipment_activity
    .filter(
        col("pickup_time").isNotNull() &
        col("delivery_time").isNotNull() &
        col("carrier_id").isNotNull()
    )
    .groupBy(
        "carrier_id",
        "carrier_name"
    )
    .agg(
        countDistinct("shipment_id").alias("delivered_shipments"),
        round(
            avg("delivery_duration_hours"),
            2
        ).alias("average_lead_time_hours")
    )
)

display(carrier_lead_time)

In [0]:
#Calculate delayed shipment count
delayed_shipments = (
    fact_shipment_activity
    .filter(
        col("delivery_performance") == "DELAYED"
    )
)

delayed_count = delayed_shipments.agg(
    countDistinct("shipment_id").alias(
        "delayed_shipment_count"
    )
)

display(delayed_count)

In [0]:
delayed_by_carrier = (
    delayed_shipments
    .groupBy("carrier_id", "carrier_name")
    .agg(
        countDistinct("shipment_id").alias(
            "delayed_shipment_count"
        )
    )
)

display(delayed_by_carrier)

In [0]:
delayed_by_carrier = (
    delayed_shipments
    .groupBy("carrier_id", "carrier_name")
    .agg(
        countDistinct("shipment_id").alias(
            "delayed_shipment_count"
        )
    )
)

display(delayed_by_carrier)

In [0]:
#Create daily shipment KPI mart
daily_data = fact_shipment_activity.withColumn(
    "delivery_date",
    to_date(col("delivery_time"))
)

mart_shipment_daily = (
    daily_data
    .filter(col("delivery_date").isNotNull())
    .groupBy("delivery_date")
    .agg(
        countDistinct("shipment_id").alias(
            "total_delivered_shipments"
        ),

        countDistinct(
            when(
                col("delivery_performance") == "ON_TIME",
                col("shipment_id")
            )
        ).alias("on_time_shipments"),

        countDistinct(
            when(
                col("delivery_performance") == "DELAYED",
                col("shipment_id")
            )
        ).alias("delayed_shipments"),

        round(
            avg("delivery_duration_hours"),
            2
        ).alias("average_lead_time_hours")
    )
)

mart_shipment_daily = mart_shipment_daily.withColumn(
    "on_time_delivery_rate",
    when(
        col("total_delivered_shipments") > 0,
        round(
            col("on_time_shipments") * 100 /
            col("total_delivered_shipments"),
            2
        )
    )
)

display(mart_shipment_daily.orderBy("delivery_date"))

In [0]:
#Create status transition fact table
fact_status_transitions = lifecycle_df.select(
    "shipment_id",
    "tracking_event_id",
    "previous_event",
    col("event_type").alias("current_event"),
    "event_ts",
    "next_event",
    "stage_duration_hours",
    "transition_status"
)

display(fact_status_transitions)

In [0]:
#Save all Gold tables
spark.sql("""
CREATE SCHEMA IF NOT EXISTS logistics.gold
""")

In [0]:
dim_carrier.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("logistics.gold.dim_carrier")

In [0]:
dim_warehouse.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("logistics.gold.dim_warehouse")

In [0]:
dim_route.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("logistics.gold.dim_route")

In [0]:
fact_shipment_activity.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("logistics.gold.fact_shipment_activity")

In [0]:
fact_status_transitions.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("logistics.gold.fact_status_transitions")

In [0]:
mart_shipment_daily.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("logistics.gold.mart_shipment_daily")

In [0]:
#Reconciliation and validation
#Check 1 – Fact table count
print(
    "Silver orders:",
    orders_df.select("shipment_id").distinct().count()
)

print(
    "Gold shipment facts:",
    fact_shipment_activity.select("shipment_id").distinct().count()
)

In [0]:
#Check 2 – Duplicate shipment IDs in fact table
display(
    fact_shipment_activity
    .groupBy("shipment_id")
    .count()
    .filter(col("count") > 1)
)

In [0]:
#Check 3 – Validate delivery performance
display(
    fact_shipment_activity.groupBy(
        "delivery_performance"
    ).count()
)

In [0]:
%sql
--Check 4 – Validate Gold tables
SHOW TABLES IN logistics.gold;

In [0]:
%sql
--delta lake evidence
DESCRIBE HISTORY logistics.gold.fact_shipment_activity;